In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

%matplotlib inline
%config InlineBackend.figure_format = 'retina'
sns.set_theme(style="whitegrid")
plt.rcParams['axes.unicode_minus'] = False

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
sample_submission = pd.read_csv('../data/sample_submission.csv')

In [3]:
from sklearn.preprocessing import LabelEncoder

train_encoded = train.copy()
test_encoded = test.copy()

cat_features = train.select_dtypes(include=['object']).columns.tolist()
if 'Irrigation_Need' in cat_features:
    cat_features.remove('Irrigation_Need')

for col in cat_features:
    le = LabelEncoder()
    full_series = pd.concat([train[col], test[col]]).astype(str).fillna('Unknown')
    le.fit(full_series)
    train_encoded[col] = le.transform(train_encoded[col].astype(str).fillna('Unknown'))
    test_encoded[col] = le.transform(test_encoded[col].astype(str).fillna('Unknown'))

print(f"Encode complete: {cat_features}")

Encode complete: ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']


In [4]:
le_target = LabelEncoder()
train_encoded['Irrigation_Need'] = le_target.fit_transform(train['Irrigation_Need'].astype(str))

target_mapping = dict(zip(le_target.classes_, le_target.transform(le_target.classes_)))
print(f"目标值映射关系: {target_mapping}")

目标值映射关系: {'High': np.int64(0), 'Low': np.int64(1), 'Medium': np.int64(2)}


In [11]:
from sklearn.ensemble import RandomForestClassifier

# 定义特征
my_features = ['Soil_Moisture', 'Mulching_Used', 'Wind_Speed_kmh', 'Temperature_C', 'Humidity']

# 训练模型
X = train_encoded[my_features]
y = train_encoded['Irrigation_Need']

model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
model.fit(X, y)

# 预测测试集
test_preds_num = model.predict(test_encoded[my_features])

# 将预测的数字转回原始文字标签
test_preds_labels = le_target.inverse_transform(test_preds_num)

import sys
sys.path.append('../../../utils')
from SubmissionHelper import save_submission

# 预测完成后直接调用
save_submission(
    preds=test_preds_labels, 
    test_df=test, 
    id_col='id',    # 对应比赛的ID列
    target_col='Irrigation_Need',    # 对应比赛的提交列
    prefix='baseline_rf'
)

------------------------------
✅ [SUCCESS] Submission file generated!
📍 Location: D:\Kaggle-Learning\02-Tabular-Data\playground-series-s6e4\submissions\baseline_rf_0404_1754.csv
📊 Shape: (270000, 2)
------------------------------


'../submissions\\baseline_rf_0404_1754.csv'